# RAG Databricks Bluetab - Creación de Tablas

Este notebook crea las tablas necesarias para el pipeline de Retrieval Augmented Generation (RAG) en Databricks.

## Tablas creadas
- **docs_text**: Almacena el texto extraído y segmentado de documentos PDF.
- **docs_track**: Lleva el seguimiento de los archivos PDF procesados para evitar duplicados.

## Características
- Creación parametrizada de tablas usando widgets
- Change Data Feed habilitado para procesamiento incremental
- ID autoincremental para los chunks de texto
- Registro de pasos y métricas en MLflow

## Dependencias
Ejecuta primero el notebook `00 Configuration and Utils` para importar variables y funciones compartidas.

In [ ]:
# Crear widgets de configuración
# Core Configuration
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")

# Table Configuration
dbutils.widgets.text("docs_text_table", "docs_text", "Documents Text Table")
dbutils.widgets.text("docs_track_table", "docs_track", "Documents Tracking Table")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
%run "./00 Configuration and Utils"

In [0]:
start_child_run("01_create_needed_tables")

## Configurar ejecución de MLflow
Inicia una ejecución de MLflow y registra parámetros relevantes como el entorno, catálogo y esquema.

In [0]:
import mlflow

# Iniciar ejecución de MLflow para este paso
mlflow.log_param("step", "table_creation")
mlflow.log_param("environment", ENVIRONMENT)
mlflow.log_param("catalog", CATALOG_NAME)
mlflow.log_param("schema", SCHEMA_NAME)

log_step("table_creation", "started", "Creando tablas fundamentales")

## Crear tabla docs_text
Define el esquema de la tabla docs_text y utiliza la función create_table_if_not_exists para crearla si no existe. Registra métricas y estados en MLflow.

In [0]:
# Definir esquema de la tabla docs_text
# Características: ID autoincremental, Change Data Feed, optimización para búsqueda y embeddings

docs_text_schema = f"""(
    id BIGINT GENERATED BY DEFAULT AS IDENTITY,
    text STRING,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    source_file STRING
) 
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
)"""

log_step("create_docs_text_table", "started", DOCS_TEXT_TABLE_FULL)

success = create_table_if_not_exists(DOCS_TEXT_TABLE_FULL, docs_text_schema)

if success:
    # Registrar estadísticas de la tabla
    table_count = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TEXT_TABLE_FULL}").collect()[0]["count"]
    mlflow.log_metric("docs_text_table_count", table_count)
    log_step("create_docs_text_table", "success", f"La tabla tiene {table_count} registros")
else:
    log_step("create_docs_text_table", "failed", "No se pudo crear la tabla")

## Crear tabla docs_track
Define el esquema de la tabla docs_track y utiliza la función create_table_if_not_exists para crearla si no existe. Registra métricas y estados en MLflow.

In [0]:
# Definir esquema de la tabla docs_track
# Características: seguimiento de archivos, timestamp, Change Data Feed

docs_track_schema = f"""(
    file_name STRING,
    processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    file_size BIGINT,
    processing_status STRING DEFAULT 'completed'
) 
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
)"""

log_step("create_docs_track_table", "started", DOCS_TRACK_TABLE_FULL)

success = create_table_if_not_exists(DOCS_TRACK_TABLE_FULL, docs_track_schema)

if success:
    table_count = spark.sql(f"SELECT COUNT(*) as count FROM {DOCS_TRACK_TABLE_FULL}").collect()[0]["count"]
    mlflow.log_metric("docs_track_table_count", table_count)
    log_step("create_docs_track_table", "success", f"La tabla tiene {table_count} registros")
else:
    log_step("create_docs_track_table", "failed", "No se pudo crear la tabla")

## Verificar tablas y registrar resultados
Verifica que ambas tablas se hayan creado correctamente usando Spark y SQL, y registra los resultados en MLflow.

In [0]:
# Verificar que ambas tablas se hayan creado correctamente y registrar resultados
try:
    # Verificar existencia de tablas
    docs_text_exists = spark.catalog.tableExists(DOCS_TEXT_TABLE_FULL)
    docs_track_exists = spark.catalog.tableExists(DOCS_TRACK_TABLE_FULL)
    # Alternativa usando SQL
    try:
        spark.sql(f"DESCRIBE {DOCS_TEXT_TABLE_FULL}")
        docs_text_exists = True
    except:
        docs_text_exists = False
    try:
        spark.sql(f"DESCRIBE {DOCS_TRACK_TABLE_FULL}")
        docs_track_exists = True
    except:
        docs_track_exists = False
    # Registrar resultados
    mlflow.log_param("docs_text_table_created", docs_text_exists)
    mlflow.log_param("docs_track_table_created", docs_track_exists)
    if docs_text_exists and docs_track_exists:
        log_step("table_verification", "success", "Todas las tablas creadas correctamente")
        mlflow.log_param("overall_status", "success")
    else:
        log_step("table_verification", "partial", f"docs_text: {docs_text_exists}, docs_track: {docs_track_exists}")
        mlflow.log_param("overall_status", "partial")
except Exception as e:
    log_step("table_verification", "error", str(e))
    mlflow.log_param("overall_status", "error")
log_step("table_creation", "completed", "Proceso de creación de tablas finalizado")

In [0]:
# Finalizar child run
try:
    log_step("table_creation", "completed", "Proceso de creación de tablas finalizado")
    end_child_run("success")
    print("✅ Child run finalizada correctamente")
except Exception as e:
    print(f"⚠️ Error finalizando child run: {e}")
    end_child_run("failed")
